In [4]:
#Implementig a perceptron
import numpy as np
import tensorflow
from sklearn.datasets import make_classification
from sklearn.linear_model import Perceptron
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from tensorflow import keras
from keras.datasets import mnist

#load mnist dataset
(X_train, y_train), (X_test, y_tesy) = mnist.load_data()

In [6]:
X_train.shape

(60000, 28, 28)

In [7]:
# Flatten 28*28 images to 784 features
x_train = X_train.reshape(X_train.shape[0], -1) / 255.0
x_test = X_test.reshape(X_test.shape[0], -1) / 255.0

In [9]:
x_train.shape

(60000, 784)

In [10]:
#train perceptron (sinlgle layer)
clf = Perceptron(max_iter=20, tol=0.001, random_state=42)
clf.fit(x_train, y_train)

d:\Programming\Deep_Learning_Concept\.tensorli\Lib\site-packages\sklearn\linear_model\_stochastic_gradient.py:726: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


,penalty,None
,alpha,0.0001
,l1_ratio,0.15
,fit_intercept,True
,max_iter,20
,tol,0.001
,shuffle,True
,verbose,0
,eta0,1.0
,n_jobs,None
,random_state,42


In [12]:
#Accuracy
acc = accuracy_score(y_tesy, clf.predict(x_test))
print(f"Accuracy: {acc}")

Accuracy: 0.8602


In [13]:
#predictions
y_pred = clf.predict(x_test)

### Shallow Neural Network


In [16]:
# Shallow NN using Tensorflow/keras
from keras.models import Sequential
from keras.layers import Dense, Flatten
from keras.datasets import mnist
from keras.utils import to_categorical

# Load MNIST dataset
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Normalize data
x_train, x_test = x_train / 255.0, x_test / 255.0

# One-hot encode labels
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

In [18]:

# Build shallow NN (1 hidden layer)
model = Sequential([
    # Flatten(input_shape=(28, 28)),     # Flatten 28x28 images
    Dense(128, activation='relu'),     # Hidden layer (shallow architecture)
    Dense(10, activation='softmax')    # Output layer
])


AttributeError: module 'keras.src.activations' has no attribute 'get'

In [ ]:
# Compile model
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

In [ ]:
# Train
history = model.fit(x_train, y_train, epochs=5, batch_size=32, validation_split=0.1)

# Evaluate
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=2)
print("\nTest accuracy:", test_acc)

### Pytorch Implementation

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt


In [ ]:
# Transform: normalize pixel values
transform = transforms.Compose([
    transforms.RandomRotation(10),
    transforms.RandomAffine(0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Load dataset
train_data = datasets.MNIST(root='./data', train=True, transform=transform, download=True)
test_data = datasets.MNIST(root='./data', train=False, transform=transform, download=True)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=1000, shuffle=False)


In [ ]:
class SquareActivation(torch.autograd.Function):
    @staticmethod
    def forward(ctx, input):
        ctx.save_for_backward(input)
        return input ** 2

    @staticmethod
    def backward(ctx, grad_output):
        (input,) = ctx.saved_tensors
        return grad_output * 2 * input

square_act = SquareActivation.apply


In [ ]:
class CustomNet(nn.Module):
    def __init__(self, hidden_activation="relu", output_unit="softmax"):
        super(CustomNet, self).__init__()
        self.fc1 = nn.Linear(28*28, 256)
        self.bn1 = nn.BatchNorm1d(256)
        self.dropout1 = nn.Dropout(0.5)

        self.fc2 = nn.Linear(256, 128)
        self.bn2 = nn.BatchNorm1d(128)
        self.dropout2 = nn.Dropout(0.5)

        self.fc3 = nn.Linear(128, 10)

        # choose hidden activation
        if hidden_activation == "tanh":
            self.hidden_act = nn.Tanh()
        elif hidden_activation == "relu":
            self.hidden_act = nn.ReLU()
        elif hidden_activation == "square":
            self.hidden_act = square_act
        else:
            raise ValueError("Invalid hidden activation")

        # choose output activation
        self.output_unit = output_unit

    def forward(self, x):
        x = x.view(-1, 28*28)
        x = self.dropout1(self.hidden_act(self.bn1(self.fc1(x))))
        x = self.dropout2(self.hidden_act(self.bn2(self.fc2(x))))
        x = self.fc3(x)

        if self.output_unit == "softmax":
            return torch.softmax(x, dim=1)
        else:  # linear output
            return x


In [ ]:
def train_model(hidden="relu", output="softmax", epochs=15, reg=1e-4):
    model = CustomNet(hidden_activation=hidden, output_unit=output)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.01, weight_decay=reg)  # weight decay = L2 regularization

    for epoch in range(epochs):
        total_loss = 0
        for images, labels in train_loader:
            optimizer.zero_grad()
            output_pred = model(images)
            loss = criterion(output_pred, labels)
            loss.backward()  # recursive chain rule → backprop
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

    # test accuracy
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    acc = 100 * correct / total
    print(f"Test Accuracy ({hidden}, {output}): {acc:.2f}%")


In [ ]:
# Different hidden activations
train_model(hidden="tanh", output="softmax", epochs=3)
train_model(hidden="relu", output="softmax", epochs=3)
train_model(hidden="square", output="softmax", epochs=3)  # custom op

# Linear output (not common for classification, but shown for experiment)
train_model(hidden="relu", output="linear", epochs=3)


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

# 1. Data (light normalization, no heavy augmentation)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_data = datasets.MNIST(root='./data', train=True, transform=transform, download=True)
test_data = datasets.MNIST(root='./data', train=False, transform=transform, download=True)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=1000, shuffle=False)


# 2. Model (shallow but strong enough)
class ShallowNet(nn.Module):
    def __init__(self):
        super(ShallowNet, self).__init__()
        self.fc1 = nn.Linear(28*28, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 10)

    def forward(self, x):
        x = x.view(-1, 28*28)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)  # raw scores → CrossEntropyLoss handles softmax
        return x


# 3. Training function
def train_model(epochs=20):
    model = ShallowNet()
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)  # no weight decay

    train_losses, val_losses, val_accs = [], [], []

    for epoch in range(epochs):
        # ---- Train ----
        model.train()
        total_loss = 0
        for images, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        train_losses.append(total_loss / len(train_loader))

        # ---- Validation ----
        model.eval()
        val_loss, correct, total = 0, 0, 0
        with torch.no_grad():
            for images, labels in test_loader:
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        val_losses.append(val_loss / len(test_loader))
        val_accs.append(100 * correct / total)

        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_losses[-1]:.4f} "
              f"| Val Loss: {val_losses[-1]:.4f} "
              f"| Val Acc: {val_accs[-1]:.2f}%")

    return model, train_losses, val_losses, val_accs


# 4. Run training
model, train_losses, val_losses, val_accs = train_model(epochs=20)


# 5. Plot curves
plt.figure(figsize=(10,4))
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Val Loss")
plt.xlabel("Epochs"); plt.ylabel("Loss"); plt.title("Loss Curve"); plt.legend()
plt.show()

plt.figure(figsize=(10,4))
plt.plot(val_accs, label="Validation Accuracy")
plt.xlabel("Epochs"); plt.ylabel("Accuracy (%)"); plt.title("Accuracy Curve"); plt.legend()
plt.show()
